# Analisis Data E-Commerce Public Dataset (Olist Brazil)

*Analisis ini dibuat dengan bahasa sehari-hari ala marketer biar gampang dipahami.*

Di notebook ini kita bakal:
1. **Gathering Data** — muat semua data CSV.
2. **Assessing Data** — cek kualitas data (Missing Value, Duplikat, Outlier, dll).
3. **Cleaning Data** — bersihkan masalah yang ketemu.
4. **EDA (Exploratory Data Analysis)** — jawab 2 pertanyaan bisnis:
   - **Q2:** Kategori produk apa yang paling laku & revenue-nya gede?
   - **Q3:** Kota mana yang jadi sumber cuan terbesar?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

folder = 'data/'

customers      = pd.read_csv(folder + 'customers_dataset.csv')
orders         = pd.read_csv(folder + 'orders_dataset.csv')
items          = pd.read_csv(folder + 'order_items_dataset.csv')
payments       = pd.read_csv(folder + 'order_payments_dataset.csv')
reviews        = pd.read_csv(folder + 'order_reviews_dataset.csv')
products       = pd.read_csv(folder + 'products_dataset.csv')
sellers        = pd.read_csv(folder + 'sellers_dataset.csv')
geolocation    = pd.read_csv(folder + 'geolocation_trimmed.csv')
category_trans = pd.read_csv(folder + 'product_category_name_translation.csv')

datasets = {
    'customers': customers,
    'orders': orders,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'category_trans': category_trans
}

for name, df in datasets.items():
    print(f'{name:15} -> shape: {df.shape}')

## Data Cleaning

- Convert timestamp columns
- Lowercase city names
- Remove duplicates


In [ ]:
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
             'order_delivered_customer_date','order_estimated_delivery_date']
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

customers['customer_city'] = customers['customer_city'].str.lower().str.strip()
geolocation = geolocation.drop_duplicates()

print('Cleaning selesai!')

## EDA - Q2: Kategori Produk Paling Laku

In [ ]:
items_products = items.merge(products[['product_id','product_category_name']], on='product_id', how='left')
items_products = items_products.merge(category_trans, on='product_category_name', how='left')
items_products['product_category_name_english'] = items_products['product_category_name_english'].fillna('unknown')

q2 = items_products.groupby('product_category_name_english').agg(
    total_items_sold = ('order_item_id', 'count'),
    total_revenue = ('price', 'sum'),
    avg_price = ('price', 'mean')
).reset_index().sort_values('total_items_sold', ascending=False)

print('Top 10 Kategori Produk:')
print(q2.head(10).to_string(index=False))

## EDA - Q3: Kota Sumber Cuan Terbesar

In [ ]:
cust_orders = customers.merge(orders[['order_id','customer_id','order_status']], on='customer_id', how='inner')
cust_orders = cust_orders[cust_orders['order_status'] == 'delivered']
cust_items = cust_orders.merge(items[['order_id','price','freight_value']], on='order_id', how='left')

q3 = cust_items.groupby('customer_city').agg(
    total_orders = ('order_id', 'nunique'),
    total_revenue = ('price', 'sum'),
    total_freight = ('freight_value', 'sum'),
    avg_order_value = ('price', 'mean')
).reset_index().sort_values('total_orders', ascending=False)

print('Top 10 Kota:')
print(q3.head(10).to_string(index=False))

## Kesimpulan

- Kategori top: bed_bath_table, health_beauty, sports_leisure
- Kota terbesar: sao paulo, rio de janeiro, belo horizonte
